In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    classify_cage_state,
)
from qlinks.basis.sectors import sector_mask_from_build_result
from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.models import SquareQuantumDiskModel
from qlinks.visualizer import (
    LinkVisualStyle,
    BasisGridVisualizer,
    LocalBasisGridVisualizer,
    HamiltonianGraphStyle,
    HamiltonianGraphVisualizer,
)

In [ ]:
model = SquareQuantumDiskModel(
    lx=4,
    ly=4,
    boundary_condition="open",
    disk_number=4,
    coup_kin=-1.0,
    coup_pot=1.0,
    chemical_potential=0.0,
    hard_core_nearest_neighbor=True,  # Hard-core nearest-neighbor blockade.
    hop_families=("x_plus_y",),  # Diagonal hopping family, "x_plus_y" means hops along (+1, -1), preserving x + y.
)

print(model.allowed_sector_labels())
print(model.nonempty_sector_labels())

In [ ]:
build_result = model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
    on_missing="raise",
)

hamiltonian_matrix = build_result.hamiltonian
kinetic_matrix = build_result.kinetic
potential_matrix = build_result.potential
basis = build_result.basis
basis_configs = basis_configs_from_build_result(build_result)
sector_mask = sector_mask_from_build_result(build_result)

print("n_states =", basis.n_states)
print("H shape =", hamiltonian_matrix.shape)
print("H nnz =", hamiltonian_matrix.nnz)
print("K nnz =", kinetic_matrix.nnz)
print("V nnz =", potential_matrix.nnz)
print("K is bipartite =", nx.is_bipartite(nx.from_scipy_sparse_array(kinetic_matrix, edge_attribute="weight")))

In [ ]:
searcher = CageSearcher.from_model_build_result(
    build_result,
    config=CageSearchConfig(
        search_type="type1",
        # type1_kappas=(0,),
        # type2_kappas=(-2, 2),
        tolerance=1e-10,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=256,
        ipr_candidate_count=128,
        ipr_random_seed=1234,
        store_full_states=True,
    ),
)

search_result = searcher.run()
print("number of cages:", len(search_result))
print("signatures:", search_result.signatures)
print("counts by signature:", search_result.counts_by_signature)

In [ ]:
signature = (0, 4)
record_index = 0
record = search_result[signature, record_index]

state_vector = record.full_state
if state_vector is None:
    state_vector = search_result.full_state_matrix()[record_index]

print("selected signature:", record.signature)
print("support size:", record.support.size)
print("energy:", record.cage_state.energy)

In [ ]:
report = classify_cage_state(
    record.cage_state,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis_configs,
    sector_mask=sector_mask,
    hilbert_size=search_result.hilbert_size,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
    ),
)

print(report)

In [ ]:
graph_visualizer = HamiltonianGraphVisualizer.from_sparse_matrix(
    kinetic_matrix,
    include_self_loops=False,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=24,
        vertex_label_size=4,
    ),
)

graph_visualizer.plot(
    backend="igraph-mpl",
    color_by="state_amplitude_real",
    edge_color_by="weight_phase",
    state_vector=state_vector,
    layout="kk",
    title=f"Fock-space graph colored by cage-state signed amplitude",
)

In [ ]:
small_viz = HamiltonianGraphVisualizer.cage_subgraph_from_sparse_matrix(
    build_result.kinetic,
    state_vector,
    classification_report=report,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=32,
        vertex_label_size=6,
    ),
)

small_viz.plot(
    backend="igraph-mpl",
    color_by="state_amplitude_real",
    edge_color_by="weight_phase",
    layout="kk",
    title=f"The relevant subgraph colored by cage-state signed amplitude",
)

In [ ]:
def draw_disk_config(model, config, *, amplitude=None, ax=None, title=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))

    data = model.basis_visualization_data(config)
    occ = data["site_occupations"]
    coords = data["site_coordinates"]

    xs_empty = []
    ys_empty = []
    xs_occ = []
    ys_occ = []

    for site_id, (x, y) in coords.items():
        if occ[site_id]:
            xs_occ.append(x)
            ys_occ.append(y)
        else:
            xs_empty.append(x)
            ys_empty.append(y)

    ax.scatter(xs_empty, ys_empty, s=120, facecolors="none", edgecolors="gray")
    ax.scatter(xs_occ, ys_occ, s=300)

    for site_id, (x, y) in coords.items():
        ax.text(x, y, str(site_id), ha="center", va="center", fontsize=8)

    ax.set_aspect("equal")
    ax.set_xticks(range(model.lx))
    ax.set_yticks(range(model.ly))
    ax.grid(True, alpha=0.3)

    if title is None:
        title = "disk config"
    if amplitude is not None:
        title += f"\namp={amplitude:.3g}"

    ax.set_title(title)
    return ax


support = record.cage_state.support
local_state = record.cage_state.local_state

n = len(support)
cols = min(4, n)
rows = int(np.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows))
axes = np.atleast_1d(axes).ravel()

for panel, (basis_index, amp) in enumerate(zip(support, local_state)):
    config = basis_configs[int(basis_index)]
    draw_disk_config(
        model,
        config,
        amplitude=amp,
        ax=axes[panel],
        title=f"basis index {basis_index}",
    )

for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()